# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

# Lab 4 - Patent Join using PySpark RDDs

The goal is to find the number of same-state citations for each patent.

I first develop the RDD solution using a 5% sample because RDD joins are
slower than the DataFrame operations. After verifying the logic on the
sample, I run the solution on the full dataset.

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Develop the solution using a sample

I use 5% of the citation and patent data while developing the RDD
solution. This makes it easier to test the joins and transformations
without waiting for the full dataset after every change.

In [6]:
rddCitationsSample = rddCitations.sample(False, 0.05)
rddPatentsSample = rddPatents.sample(False, 0.05)

In [7]:
rddCitationsSample.take(5)

['3858245,2072303',
 '3858247,2807431',
 '3858263,3608118',
 '3858275,3131364',
 '3858287,1687498']

In [8]:
rddPatentsSample.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070823,1963,1096,,"US","MI",,1,,401,1,12,,8,,0.6563,,,,,,,',
 '3070842,1963,1096,,"US","TX",,1,,425,5,51,,1,,0,,,,,,,',
 '3070844,1963,1096,,"US","OH",,2,,425,5,51,,2,,0,,,,,,,',
 '3070905,1963,1096,,"US","NJ",,2,,434,6,69,,0,,,,,,,,,']

In [11]:
patent_pairs = rddPatentsSample.map(
    lambda line: line.split(",")
).filter(
    lambda fields: fields[0] != '"PATENT"'
).filter(
    lambda fields: len(fields) > 5
).map(
    lambda fields: (
        int(fields[0]),
        fields[5].replace('"', '') if fields[5] else None
    )
)

In [12]:
patent_pairs.take(5)

[(3070823, 'MI'),
 (3070842, 'TX'),
 (3070844, 'OH'),
 (3070905, 'NJ'),
 (3070916, 'WA')]

In [13]:
citation_pairs = rddCitationsSample.map(
    lambda line: line.split(",")
).map(
    lambda fields: (int(fields[1]), int(fields[0]))
)

In [14]:
citation_pairs.take(5)

[(2072303, 3858245),
 (2807431, 3858247),
 (3608118, 3858263),
 (3131364, 3858275),
 (1687498, 3858287)]

In [15]:
cited_joined = citation_pairs.join(patent_pairs)

In [16]:
cited_joined.take(5)

[(3707128, (3858533, 'TX')),
 (3272218, (3858607, 'CA')),
 (3272218, (4014510, 'CA')),
 (3696868, (3858650, 'TX')),
 (3696868, (3976136, 'TX'))]

In [17]:
citing_pairs = rddCitationsSample.map(
    lambda line: line.split(",")
).map(
    lambda fields: (int(fields[0]), int(fields[1]))
)

In [18]:
citing_pairs.take(5)

[(3858245, 2072303),
 (3858247, 2807431),
 (3858263, 3608118),
 (3858275, 3131364),
 (3858287, 1687498)]

In [19]:
citing_joined = citing_pairs.join(patent_pairs)

In [20]:
citing_joined.take(5)

[(3858362, (2722092, 'NY')),
 (3858364, (2307899, 'NH')),
 (3858864, (2803872, 'FL')),
 (3859000, (3235040, 'VA')),
 (3859018, (2591546, 'CO'))]

In [21]:
joined = citing_joined.join(cited_joined)

In [22]:
joined.take(10)

[(3859000, ((3235040, 'VA'), (4963055, 'VA'))),
 (3859000, ((3235040, 'VA'), (5097643, 'VA'))),
 (3859000, ((3235040, 'VA'), (5295341, 'VA'))),
 (3859846, ((3204457, ''), (3954119, ''))),
 (3860526, ((3435618, ''), (5698109, ''))),
 (3862738, ((3664581, ''), (4482127, ''))),
 (3864522, ((3702902, 'CA'), (5960337, 'CA'))),
 (3866136, ((3320533, 'IL'), (4521912, 'IL'))),
 (3866136, ((3320533, 'IL'), (5995851, 'IL'))),
 (3866342, ((3648782, 'NY'), (5129169, 'NY')))]

In [23]:
same_state = joined.filter(
    lambda x: x[1][0][1] != '' and
              x[1][1][1] != '' and
              x[1][0][1] == x[1][1][1]
)

In [24]:
same_state.take(10)

[(3859000, ((3235040, 'VA'), (4963055, 'VA'))),
 (3859000, ((3235040, 'VA'), (5097643, 'VA'))),
 (3859000, ((3235040, 'VA'), (5295341, 'VA'))),
 (3864522, ((3702902, 'CA'), (5960337, 'CA'))),
 (3866136, ((3320533, 'IL'), (4521912, 'IL'))),
 (3866136, ((3320533, 'IL'), (5995851, 'IL'))),
 (3866342, ((3648782, 'NY'), (5129169, 'NY'))),
 (3866342, ((3648782, 'NY'), (5315772, 'NY'))),
 (3867950, ((3638656, 'MD'), (3987799, 'MD'))),
 (3869698, ((3519991, 'PA'), (4691239, 'PA')))]

In [25]:
same_state_counts = same_state.map(
    lambda x: (x[0], 1)
).reduceByKey(
    lambda a, b: a + b
)

In [26]:
same_state_counts.take(10)

[(3859000, 3),
 (3864522, 1),
 (3866136, 2),
 (3866342, 2),
 (3867950, 1),
 (3869698, 1),
 (3872050, 3),
 (3874480, 1),
 (3875488, 2),
 (3875494, 1)]

In [27]:
top10 = same_state_counts.sortBy(
    lambda x: x[1],
    ascending=False
).take(10)

In [28]:
top10

[(5111638, 80),
 (4748669, 42),
 (5176668, 35),
 (5554110, 30),
 (4723936, 24),
 (5672198, 20),
 (4989607, 20),
 (5298919, 20),
 (4671851, 18),
 (4716585, 18)]

## Full dataset solution

After testing the algorithm on the sample, I run the corrected RDD
solution on the full dataset.

Each citation is given a unique ID so that the state of the citing patent
and the state of the cited patent can be attached to the same original
citation without mixing different citation records.

In [29]:
rddCitationsFull = rddCitations
rddPatentsFull = rddPatents

In [30]:
patent_pairs_full = rddPatentsFull.map(
    lambda line: line.split(",")
).filter(
    lambda fields: fields[0] != '"PATENT"'
).filter(
    lambda fields: len(fields) > 5
).map(
    lambda fields: (
        int(fields[0]),
        fields[5].replace('"', '') if fields[5] else None
    )
)

In [31]:
patent_pairs_full.take(5)

[(3070801, ''),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

In [32]:
citation_records = rddCitationsFull.map(
    lambda line: line.split(",")
).filter(
    lambda fields: fields[0] != '"CITING"'
).map(
    lambda fields: (int(fields[0]), int(fields[1]))
)

In [33]:
citation_records.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

In [34]:
patent_lookup = patent_pairs_full

In [35]:
citation_records_id = citation_records.zipWithIndex()

citation_records_id.take(5)

[((3858241, 956203), 0),
 ((3858241, 1324234), 1),
 ((3858241, 3398406), 2),
 ((3858241, 3557384), 3),
 ((3858241, 3634889), 4)]

### Find the state of the citing patent

The citation ID is preserved while looking up the citing patent. This
allows the result to be connected back to the exact original citation.

In [36]:
citing_state_by_id = citation_records_id.map(
    lambda x: (x[0][0], (x[1], x[0][1]))
).join(
    patent_lookup
).map(
    lambda x: (
        x[1][0][0],
        (x[0], x[1][0][1], x[1][1])
    )
)

citing_state_by_id.take(5)

[(1008, (3858442, 213525, '')),
 (1009, (3858442, 642978, '')),
 (1010, (3858442, 1937115, '')),
 (1011, (3858442, 2187345, '')),
 (1012, (3858442, 3193035, ''))]

### Find the state of the cited patent

The same citation ID is preserved while looking up the cited patent.

In [37]:
cited_state_by_id = citation_records_id.map(
    lambda x: (x[0][1], (x[1], x[0][0]))
).join(
    patent_lookup
).map(
    lambda x: (
        x[1][0][0],
        (x[1][0][1], x[1][1])
    )
)

cited_state_by_id.take(5)

[(877, (3858416, 'CA')),
 (524709, (3968663, 'CA')),
 (687085, (4001562, 'CA')),
 (945258, (4052599, 'CA')),
 (1923730, (4237802, 'CA'))]

### Combine the two state lookups

Both results are keyed by the unique citation ID, so they can be joined
without creating combinations between unrelated citation records.

In [38]:
citation_with_states_full = citing_state_by_id.join(
    cited_state_by_id
)

citation_with_states_full.take(10)

[(5780620, ((4828786, 3372429, 'NY'), (4828786, ''))),
 (7013936, ((4993114, 4688297, ''), (4993114, ''))),
 (8912276, ((5233940, 3221708, 'MN'), (5233940, 'CA'))),
 (11985144, ((5571156, 4715381, 'MD'), (5571156, ''))),
 (12692932, ((5641310, 4902238, 'CT'), (5641310, 'CA'))),
 (13346740, ((5705494, 4943566, ''), (5705494, ''))),
 (13809296, ((5751444, 3630612, 'CA'), (5751444, 'MA'))),
 (14766212, ((5844406, 4947470, 'PA'), (5844406, 'TX'))),
 (15307968, ((5896290, 5465215, ''), (5896290, 'OH'))),
 (2704640, ((4369248, 3761278, ''), (4369248, '')))]

### Filter same-state citations

I keep only citations where both states are present and the citing and
cited states are equal.

In [39]:
same_state_full = citation_with_states_full.filter(
    lambda x: x[1][0][2] not in ('', None) and
              x[1][1][1] not in ('', None) and
              x[1][0][2] == x[1][1][1]
)

same_state_full.take(10)

[(6932956, ((4981978, 4629770, 'TX'), (4981978, 'TX'))),
 (15998996, ((5961330, 5035619, 'CA'), (5961330, 'CA'))),
 (440100, ((3951348, 3272446, 'WI'), (3951348, 'WI'))),
 (12822248, ((5653842, 4946454, 'WI'), (5653842, 'WI'))),
 (4697052, ((4677342, 3096457, 'MA'), (4677342, 'MA'))),
 (1517696, ((4164682, 3943399, 'IL'), (4164682, 'IL'))),
 (2322780, ((4305736, 4008057, 'PA'), (4305736, 'PA'))),
 (3256284, ((4458774, 3918540, 'WI'), (4458774, 'WI'))),
 (6708720, ((4952008, 4734831, 'MI'), (4952008, 'MI'))),
 (6968096, ((4986808, 3311352, 'NY'), (4986808, 'NY')))]

### Count same-state citations

Each remaining citation represents one valid same-state citation. I use
reduceByKey to add 1 for every matching citation for each citing patent.

In [40]:
same_state_counts_full = same_state_full.map(
    lambda x: (x[1][0][0], 1)
).reduceByKey(
    lambda a, b: a + b
)

same_state_counts_full.take(10)

[(5463848, 8),
 (4952008, 8),
 (4563408, 10),
 (5891380, 36),
 (4352396, 3),
 (4185004, 2),
 (5975132, 5),
 (5417332, 3),
 (5201960, 3),
 (5873852, 13)]

## Final result

I sort the full-data counts in descending order and select the top 10
patents with the most same-state citations.

In [41]:
top10_rdd = same_state_counts_full.sortBy(
    lambda x: x[1],
    ascending=False
).take(10)

top10_rdd

[(5959466, 125),
 (5983822, 103),
 (6008204, 100),
 (5952345, 98),
 (5958954, 96),
 (5998655, 96),
 (5936426, 94),
 (5739256, 90),
 (5980517, 90),
 (5978329, 90)]